In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install torch-pruning
!pip install torchinfo

In [ ]:
import torch
import torch_pruning as tp
from torchvision import models
import inspect
import json # To save metrics
import torch.nn as nn

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
num_classes = 1  

model = models.regnet_y_800mf(weights=models.RegNet_Y_800MF_Weights.DEFAULT)

FEATURE_DIM = model.fc.in_features 
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(FEATURE_DIM, 128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, num_classes) 
)



state = torch.load("/content/state_dict_RGY.pth", map_location=device) #<--- change this to your state_dict.pth model

model.load_state_dict(state)
model = model.to(device)
model.eval()

print("Model loaded successfully!")

In [ ]:
print("---------------- before pruning ---------------- ")
from torchinfo import summary
summary(model, input_size=(1, 3, 224, 224))


In [ ]:
print("---------------- before pruning ---------------- ")
from torchsummary import summary
original_model = summary(model, (3, 224, 224))


In [ ]:
print("---------------- before pruning ---------------- ")
from torchinfo import summary
summary(model, input_size=(1,3,224,224))

In [ ]:
with torch.no_grad():
    x = torch.randn(1,3,224,224).to(device)
    y = model(x)
    print(y.shape)


In [ ]:
# 1) Shapes
print("Conv shapes before pruning:")
for name, m in model.named_modules():
    if isinstance(m, torch.nn.Conv2d):
        print(name, m.weight.shape)

In [ ]:
total = sum(p.numel() for p in model.parameters())
nonzero = sum(torch.count_nonzero(p).item() for p in model.parameters())
print("Total params:", total)
print("Nonzero params:", nonzero)
print("Zero fraction:", 1 - nonzero/total)



# **Pruning**


In [ ]:
!pip show torch-pruning

In [ ]:
example_inputs = torch.randn(1, 3, 224, 224).to(device)

In [ ]:
conv_list = []
for name, m in model.named_modules():
    if isinstance(m, torch.nn.Conv2d):
        conv_list.append((name, m))
print(f"Total Conv2d modules found: {len(conv_list)}")
print("Sample conv modules (name, in_c, out_c, k, groups):")
for name, m in conv_list[:20]:
    print(f"{name:40s} | in={m.in_channels:4d} out={m.out_channels:4d} k={m.kernel_size} g={m.groups}")

In [ ]:
# List all Conv2d modules with their names
conv_list = [(name, m) for name, m in model.named_modules() if isinstance(m, torch.nn.Conv2d)]

# find candidate 1x1 convs (groups==1 and kernel 1x1) =
prunable_candidates = []
for name, m in conv_list:
    if m.kernel_size == (1, 1) and m.groups == 1:
        prunable_candidates.append((name, m))

print(f"Prunable (1x1 conv, groups==1) candidates: {len(prunable_candidates)}")
for name, m in prunable_candidates:
    print(f" - {name}: in={m.in_channels} out={m.out_channels}")


In [ ]:
# find depthwise convs and linear layers (ignored candidates) 
ignored = []
for name, m in model.named_modules():
    # depthwise convs (groups == in_channels) → in RegNetY-800MF, none by default
    if isinstance(m, torch.nn.Conv2d) and m.groups == m.in_channels:
        ignored.append((name, m))
    # linear layers
    if isinstance(m, torch.nn.Linear):
        ignored.append((name, m))

print(f"Ignored modules (depthwise convs + linears): {len(ignored)}")
for name, m in ignored[:30]:
    print(" -", name, type(m), getattr(m, "in_channels", None), getattr(m, "out_features", None))


In [ ]:
from collections import Counter

types_count = Counter(type(m) for n,m in model.named_modules())
print(types_count)


In [ ]:
print("Device:", device)
print("Total params (before):", tp.utils.count_params(model))


In [ ]:
importance = tp.importance.MagnitudeImportance(p=2) #L2 norm

pruner = tp.pruner.MagnitudePruner(
    model=model,
    example_inputs=example_inputs,
    importance=importance,
    pruning_ratio=0.4, # ----------------- percentage of pruning
    ignored_layers=[m for _, m in ignored],
)

In [ ]:
print("Pruner created. Will run step() now.")
before_params = tp.utils.count_params(model)
pruner.step()
after_params = tp.utils.count_params(model)
print("Params before:", before_params)
print("Params after: ", after_params)


In [ ]:
print(" --------------------- after pruning ----------------------")
from torchinfo import summary
summary(model, input_size=(1, 3, 224, 224))


In [ ]:
print("---------------- after pruning ---------------- ")
from torchsummary import summary
original_model = summary(model, (3, 224, 224))


In [ ]:
# model

In [ ]:
# 1) Shapes
print("Conv shapes after pruning:")
for name, m in model.named_modules():
    if isinstance(m, torch.nn.Conv2d):
        print(name, m.weight.shape)

# 2) Param and nonzero counts
total = sum(p.numel() for p in model.parameters())
nonzero = sum(torch.count_nonzero(p).item() for p in model.parameters())
print("Total params:", total)
print("Nonzero params:", nonzero)
print("Zero fraction:", 1 - nonzero/total)



In [ ]:
# ensures the pruning didn’t break the computation graph.
with torch.no_grad():
    x = torch.randn(1, 3, 224, 224).to(device)
    y = model(x)
    print("Output shape:", y.shape)


#Fine-tuning

In [ ]:
import os
# ------------------------ for the splitting of dataset ------------------------
import random
from pathlib import Path
# ------------------------for the lebling of dataset ------------------------
from torchvision import datasets, transforms
from torch.utils.data import DataLoader #turn the dataset into iterable and also for batching
from PIL import Image
from typing import List
# ------------------------ for the training of the model as MobileNetV2 ------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models  # ------------------ For MobileNetV2 ( or any pretrained model )
from sklearn.metrics import precision_score, recall_score, f1_score
# ------------------------ for plotting the curves ------------------------
import matplotlib.pyplot as plt
# ------------------------ for getting the summary of the model ------------------------
from torchinfo import summary

In [ ]:
def walk_through_dir(dir_path):
    for dirpath, dirname, filenames in os.walk(dir_path):
        print(f"There are {len(dirname)} directories and {len(filenames)} images in '{dirpath}'")


dataset_path = "/content/drive/MyDrive/colon_after_splitting" # or change the daraset and the train_transform
# dataset_path = "/content/drive/MyDrive/COVID19+PNEUMONIA+NORMAL Chest X-Ray Image Dataset"
# dataset_path = "/content/drive/MyDrive/K+F MRI"
walk_through_dir(dataset_path)  

In [ ]:
train_dir = os.path.join(dataset_path, "train")
val_dir   = os.path.join(dataset_path, "val")

In [ ]:
train_transform = transforms.Compose([

    transforms.Resize (size = (224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  
                         std=[0.229, 0.224, 0.225]),

])

val_transform = transforms.Compose([ 
  transforms.Resize (size = (224, 224)),
   transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406],  
                         std=[0.229, 0.224, 0.225]),
 ])


In [ ]:
# # for the MRI & X-ray datasets
# from PIL import Image

# train_transform = transforms.Compose([
#     transforms.Lambda(lambda img: img.convert("RGB")),   # <--- ensures 3 channels always
#     transforms.Resize((224, 224)),
#     transforms.RandomHorizontalFlip(p=0.5),
#     transforms.RandomRotation(degrees=10),

#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225]),
# ])


# val_transform = transforms.Compose([
#     transforms.Lambda(lambda img: img.convert("RGB")),
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                         std=[0.229, 0.224, 0.225]),
# ])

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform = train_transform)
val_dataset   = datasets.ImageFolder(val_dir, transform = val_transform)

print("Original class mapping:", train_dataset.class_to_idx)
print("---------------------------------------------------------")
print (train_dataset)
print("=====================================================")
print (val_dataset)

In [ ]:
print("traine dataset classes     :", train_dataset.class_to_idx, "| number of samples:", len(train_dataset))
print("validation dataset classes :", val_dataset.class_to_idx, "| number of samples:", len(val_dataset))

In [ ]:
batch_size = 32
n_workers = 0

train_loader = DataLoader(train_dataset,batch_size=batch_size, shuffle=True, num_workers = n_workers) #----- Training: “Mix it up so the model learns broadly.”
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")

print("============================================")

print(f"length of train_loader : {len(train_loader)} batches of  {batch_size}")
print(f"length of val_loader   : {len(val_loader)} batches of  {batch_size}")


In [ ]:
from timeit import default_timer as timer
def print_train_time (start : float,
                      end : float,
                      device : torch.device = None):

    """ print diffrence between start and end time. """

    total_time = end - start
    print(f"Total training time on {device}: {total_time:.3f} seconds")
    return total_time


#* calculating time it takes to train the model

In [ ]:
print("Params after pruning:", tp.utils.count_params(model))


In [ ]:
# Freeze nothing — allow training to readjust
for param in model.parameters():
    param.requires_grad = True

In [ ]:
criterion = nn.BCEWithLogitsLoss()  # Binary Cross Entropy Loss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4) #smaller LR

In [ ]:
from torchinfo import summary
summary(model, input_size=(1, 3, 224, 224))


In [ ]:
from tqdm.auto import tqdm  # for progress bar

torch.manual_seed(42)
start_time = timer() # start timing
pos = 0 #colon: 0, Xray / MRI: 1
num_epochs = 5

#?---------------------------- save metrics ----------------------------
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
train_precisions, train_recalls, train_f1s = [], [], []
val_precisions, val_recalls, val_f1s = [], [], []
#%---------------------------- Training Loop ----------------------------
for epoch in range(num_epochs):
    model.train() #$the model use the batchnorm and dropout layers

    running_loss, correct, total = 0.0, 0, 0

    all_labels, all_preds = [], []


    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
        inputs, labels = inputs.to(device), labels.to(device)

#----------------------- train step -----------------------
        outputs = model(inputs) # &----------------------- forward pass

        labels = labels.float().unsqueeze(1)
        loss = criterion(outputs, labels) #& ------------- compute the loss (predicytions vs true labels

        optimizer.zero_grad() #& --------------------------clear the gradients, set it to zero so it start fresh for this batch/ loop
        loss.backward() #& ------------------------------- backward pass
        optimizer.step() #& ------------------------------update the weights

    #= collect loss & metrics

        probs = torch.sigmoid(outputs) #convert logits to probabilities
        predicted = (probs > 0.5).long()   #convert probabilities to binary

        running_loss += loss.item() * inputs.size(0)
        correct += (predicted == labels.long()).sum().item()
        total += labels.size(0)

#?----------------------- collect the predictions and true labels

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy()) #collect true lebles and prediction across all batches

#*----------------------- compute metrics for training (storing them in the lists) -----------------------
    train_loss = running_loss / len(train_dataset)
    train_acc = correct / total
    train_precision = precision_score(all_labels, all_preds, average='binary', pos_label=pos)
    train_recall = recall_score(all_labels, all_preds, average='binary', pos_label=pos)
    train_f1 = f1_score(all_labels, all_preds, average='binary', pos_label=pos)

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    train_precisions.append(train_precision)
    train_recalls.append(train_recall)
    train_f1s.append(train_f1)

    #$ --------------------------------------------- Validation Loop ---------------------------------------------
    model.eval() #$---------------------------- note : the model.eval disable dropout and batchnorm layers
    val_running_loss, val_correct, val_total = 0.0,0 ,0
    val_labels_all, val_preds_all = [], []

    with torch.no_grad():

        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
            inputs, labels = inputs.to(device), labels.to(device)


            #only calculate the forward pass and the loss for the validation set

            outputs = model(inputs)

            labels = labels.float().unsqueeze(1)  # shape [B,1], Adds a new dimension of size 1, to model outputs,which usually are [batch_size, 1] for binary classification.
            loss = criterion(outputs, labels)

            probs = torch.sigmoid(outputs)
            predicted = (probs > 0.5).long()

            val_running_loss += loss.item() * inputs.size(0)
            val_correct += (predicted == labels.long()).sum().item()
            val_total += labels.size(0)

            val_labels_all.extend(labels.cpu().numpy())
            val_preds_all.extend(predicted.cpu().numpy())

    val_loss = val_running_loss / len(val_dataset)
    val_acc = val_correct / val_total
    val_precision = precision_score(val_labels_all, val_preds_all, average='binary', pos_label=pos)
    val_recall = recall_score(val_labels_all, val_preds_all, average='binary', pos_label=pos)
    val_f1 = f1_score(val_labels_all, val_preds_all, average='binary', pos_label=pos)

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_precisions.append(val_precision)
    val_recalls.append(val_recall)
    val_f1s.append(val_f1)

       # Save metrics after each epoch
    metrics = {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_accuracies": train_accuracies,
        "val_accuracies": val_accuracies,
        "train_precisions": train_precisions,
        "train_recalls": train_recalls,
        "train_f1s": train_f1s,
        "val_precisions": val_precisions,
        "val_recalls": val_recalls,
        "val_f1s": val_f1s,
    }
    with open("training_metrics.json", "w") as f:
        json.dump(metrics, f)



#*---------------------------- Print epoch metrics ----------------------------
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.5f}, Acc: {train_acc*100:.2f}%, Precision: {train_precision:.4f}, Recall: {train_recall:.4f}, F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.5f}, Acc: {val_acc*100:.2f}%, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
    print("-----------------------------------------------------------")

end_time = timer() # end timing
training_time = print_train_time(start_time,end_time, device)

In [ ]:
with open("training_metrics.json","r") as f:
    metrics = json.load(f)

train_losses = metrics["train_losses"]
val_losses = metrics["val_losses"]


# **Section #4**: Visulize

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Val Accuracy")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()


plt.show()

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_precisions, label="Train Precision")
plt.plot(val_precisions, label="Val Precision")
plt.xlabel('Epochs')
plt.ylabel('Precision')
plt.title('Precision')
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.show()


In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_recalls, label="Train Recall")
plt.plot(val_recalls, label="Val Recall")
plt.xlabel('Epochs')
plt.ylabel('Recall')
plt.title('Recall')
plt.legend()
plt.grid(True)

plt.show()


In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_f1s, label="Train F1")
plt.plot(val_f1s, label="Val F1")
plt.xlabel('Epochs')
plt.ylabel('F1 Score')
plt.title('F1 Score')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
class_names = train_dataset.classes
print(f"Class names: {class_names}")

In [ ]:
import torch

# Define the paths to save the model to the Kaggle output directory
save_path_state_dict = "/content/drive/MyDrive/models/Pruning/pruned_RGY_tate_dict.pth"
save_path_entire_model = "/content/drive/MyDrive/models/Pruning/pruned_RGY_entire.pth"


torch.save(model.state_dict(), save_path_state_dict)
print(f"Model state dictionary saved to {save_path_state_dict}")

torch.save(model, save_path_entire_model)
print(f"Entire model saved to {save_path_entire_model}")